# Tutorial 04b: Logistic Regression & Regularization (Lasso & Ridge)

Mathematics for Machine Learning (Mathe3)

Thomas Manke  
2026-11-10

Our goal is to build a multivariable classifier to predict the presence
of coronary heart disease ($y \in \{0, 1\}$) from clinical,
physiological, and angiographic screening measurements.

This tutorial spans **two 90-minute lab sessions (2 weeks)** and bridges
linear models to classification and regularization:

- **Part I (Session 1 — 90 min): From Linear Regression to Logistic
  Regression**
  - **Task 1: Streamlined Data Preparation** — Target encoding
    verification, class imbalance audit, stratified partitioning, and
    feature standardization ($Z$-scoring).
  - **Task 2: Linear Regression Baseline (Linear Probability Model)** —
    Fitting Ordinary Least Squares directly on a binary target to expose
    its fundamental failure modes (out-of-bounds predictions, extreme
    outlier sensitivity, and residual heteroscedasticity).
  - **Task 3: The Logistic Link Function & Multivariable Fitting** —
    Deriving the Sigmoid activation, log-odds (logit) link, Binary
    Cross-Entropy loss, and gradient connection to OLS.
  - **Task 4: Odds Ratios & Parameter Interpretation** — Translating
    weights into multiplicative odds multipliers
    ($\text{OR}_j = \exp(\theta_j)$) and auditing cardiac risk factors.
  - **Task 5: Decision Boundaries & Diagnostic Evaluation** — Mapping
    continuous probabilities to discrete decisions, constructing the
    $2 \times 2$ Confusion Matrix, analyzing Sensitivity vs. Specificity
    trade-offs, and generating ROC-AUC curves.
- **Part II (Session 2 — 90 min): Regularization Mechanics, Ridge
  vs. Lasso & Tuning**
  - **Task 6: Multicollinearity Pathology & Overfitting Audit** —
    Understanding why unconstrained optimization breaks down when
    features are redundant and correlated.
  - **Task 7: Ridge ($L_2$) Regularization** — Smooth weight shrinkage,
    Hessian conditioning
    ($\mathbf{X}^T\mathbf{W}\mathbf{X} + \lambda\mathbf{I}$), and
    variance reduction.
  - **Task 8: Lasso ($L_1$) Regularization** — Diamond ball geometry,
    corner solutions, sparsity, and automated clinical biomarker
    selection.
  - **Task 9: Regularization Paths Across 6 Orders of Magnitude** —
    Tracing continuous weight trajectories $\theta_j(C)$ and sparsity
    profiles as penalty strength varies.
  - **Task 10: 5-Fold Stratified Cross-Validation & Hyperparameter
    Tuning** — Systematically selecting optimal penalty parameters using
    `LogisticRegressionCV`.
  - **Task 11: Final Out-of-Sample Evaluation & Model Critique** —
    Comparing Linear vs. Unregularized vs. Ridge vs. Lasso models on
    unseen test data.

------------------------------------------------------------------------

# Part I: From Linear Regression to Logistic Regression (Session 1 — 90 min)

## Setup

Run the setup cell below to verify package installations, configure
visualization aesthetics, and load the dataset.

- **Dataset Origin:** The
  <a href="https://archive.ics.uci.edu/dataset/45/heart+disease"
  target="_blank" rel="noopener">Cleveland Heart Disease Dataset</a> was
  compiled by the Cleveland Clinic Foundation (Detrano et al., 1989).
- **Feature Dimensionality:** 13 clinical, electrocardiographic, and
  angiographic diagnostic indicators:
  - `age`: Patient age (years)
  - `sex`: Biological sex ($1 = \text{male}, 0 = \text{female}$)
  - `cp`: Chest pain type
    ($1 = \text{typical angina}, 2 = \text{atypical angina}, 3 = \text{non-anginal pain}, 4 = \text{asymptomatic}$)
  - `trestbps`: Resting blood pressure ($\text{mm Hg}$ on hospital
    admission)
  - `chol`: Serum cholesterol ($\text{mg/dL}$)
  - `fbs`: Fasting blood sugar $> 120\text{ mg/dL}$
    ($1 = \text{true}, 0 = \text{false}$)
  - `restecg`: Resting electrocardiographic results
    ($0 = \text{normal}, 1 = \text{ST-T wave abnormality}, 2 = \text{left ventricular hypertrophy}$)
  - `thalach`: Maximum heart rate achieved during exercise stress
    testing
  - `exang`: Exercise-induced angina ($1 = \text{yes}, 0 = \text{no}$)
  - `oldpeak`: ST depression induced by exercise relative to rest
    ($\text{mm}$)
  - `slope`: Slope of the peak exercise ST segment
    ($1 = \text{upsloping}, 2 = \text{flat}, 3 = \text{downsloping}$)
  - `ca`: Number of major coronary vessels ($0\text{–}3$) colored by
    fluoroscopy
  - `thal`: Thallium scintigraphy heart scan
    ($3 = \text{normal}, 6 = \text{fixed defect}, 7 = \text{reversible defect}$)

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, log_loss, accuracy_score

sns.set_theme(context="notebook", style="whitegrid")
FIGSIZE = (7, 4)
RANDOM_STATE = 42

print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")

# Load Cleveland Heart Disease Dataset
df = None
for path in ["../../data/heart_disease.csv", "../data/heart_disease.csv", "data/heart_disease.csv"]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        break

if df is None:
    url = "https://raw.githubusercontent.com/thomasmanke/Mathe3/main/data/heart_disease.csv"
    df = pd.read_csv(url)

print(f"Dataset shape: {df.shape} (n = {df.shape[0]} patients, {df.shape[1]-1} features + target)")
df.head()

Python Version: 3.11.16
NumPy Version: 2.4.6
Pandas Version: 3.0.5
Scikit-Learn Version: 1.9.1
Dataset shape: (297, 14) (n = 297 patients, 13 features + target)

------------------------------------------------------------------------

## Task 1: Streamlined Data Preparation

A sound classification workflow requires rigorous data preparation
before mathematical estimation:

- **Target Encoding:** The binary target indicates presence of coronary
  artery disease:
  $$y_i = \begin{cases} 1 & \text{if Coronary Disease Present} \\ 0 & \text{if Healthy} \end{cases}$$
- **Class Imbalance Audit:** Calculate sample counts and empirical
  disease prevalence:
  $$\text{Prevalence} = \frac{1}{n}\sum_{i=1}^n y_i$$ *Reflection:* How
  balanced is this clinical cohort? What baseline accuracy would a naive
  majority guesser achieve?
- **Stratified Partitioning:** Split the dataset into 80% training
  ($n_{\text{train}} = 237$) and 20% test ($n_{\text{test}} = 60$)
  subsets using `stratify=y` to preserve exact class proportions in both
  sets.
- **Feature Standardization ($Z$-scoring):** Standardize all 13
  predictors by centering to zero mean and scaling to unit variance:
  $$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$ *Methodological Rule:* Fit
  the `StandardScaler` strictly on the training partition, then
  transform both training and test partitions to prevent data leakage.

------------------------------------------------------------------------

## Task 2: Linear Regression Baseline & Exposing Its Failure Modes

Before implementing logistic regression, we examine what happens if we
apply Ordinary Least Squares (OLS) linear regression—the **Linear
Probability Model (LPM)**—directly to a qualitative binary target:
$$y_i = \mathbf{x}_i^T \boldsymbol{\theta} + \varepsilon_i, \quad y_i \in \{0, 1\}$$

- **Bivariate OLS Fit:** Select a prominent physiological predictor,
  `oldpeak` (exercise-induced ST depression, $x$), and fit a simple 1D
  linear regression model $\hat{y} = \theta_0 + \theta_1 x$.
- **Visualizing the Linear Probability Fit:** Plot the fitted line
  $\hat{y}(x)$ against the binary scatter points $y \in \{0, 1\}$.
- **Failure Mode 1 (Out-of-Bounds Predictions):** Identify data points
  where the linear model predicts $\hat{y} < 0$ or $\hat{y} > 1$.
  *Conceptual Question:* What does a predicted probability of
  $\hat{y} = -0.12$ or $\hat{y} = 1.20$ mean clinically? Why does this
  violate Kolmogorov’s probability axioms?
- **Failure Mode 2 (Extreme Outlier Distortion):** Introduce a patient
  with an extremely high ST depression ($x = 5.0\text{ mm}$, unequivocal
  heart disease $y=1$). Refit the OLS model. Observe how squared error
  $(y_i - \hat{y}_i)^2$ severely penalizes this extreme point even
  though it is correctly classified, tilting the line and shifting the
  decision threshold ($\hat{y} = 0.5$), which misclassifies borderline
  patients.
- **Failure Mode 3 (Violation of Gauss-Markov Homoscedasticity):** In
  OLS, we assume $\operatorname{Var}(\varepsilon \mid X) = \sigma^2$.
  However, for a Bernoulli variable $Y \in \{0, 1\}$ with conditional
  expectation $p(x) = \mathbb{E}[Y \mid X]$:
  $$\operatorname{Var}(Y \mid X) = p(x)(1 - p(x)) = (\mathbf{x}^T\boldsymbol{\theta})(1 - \mathbf{x}^T\boldsymbol{\theta})$$
  The variance inherently varies with $x$, completely invalidating
  standard OLS hypothesis testing and standard errors.

------------------------------------------------------------------------

## Task 3: The Logistic Link Function & Multivariable Fitting

To resolve the fatal flaws of the Linear Probability Model, we model the
probability via a smooth non-linear mapping from
$\mathbb{R} \to (0, 1)$:

### Mathematical Formulation

1.  **Linear Predictor:**
    $$z_i = \mathbf{x}_i^T \boldsymbol{\theta} = \theta_0 + \sum_{j=1}^p \theta_j z_{ij}$$
2.  **Sigmoid Activation (Logistic Link):**
    $$\hat{p}_i = P(y_i = 1 \mid \mathbf{x}_i) = \sigma(z_i) = \frac{1}{1 + e^{-z_i}} = \frac{1}{1 + e^{-\mathbf{x}_i^T \boldsymbol{\theta}}}$$
    The sigmoid function satisfies $\lim_{z \to -\infty} \sigma(z) = 0$
    and $\lim_{z \to +\infty} \sigma(z) = 1$, strictly bounding
    predictions in $(0, 1)$ while plateauting for extreme confident
    values.
3.  **Log-Odds (Logit):**
    $$\ln\left(\frac{p_i}{1 - p_i}\right) = z_i = \mathbf{x}_i^T \boldsymbol{\theta}$$
4.  **Binary Cross-Entropy Loss (Negative Log-Likelihood):**
    $$\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{n}\sum_{i=1}^n \left[ y_i \ln(\hat{p}_i) + (1 - y_i) \ln(1 - \hat{p}_i) \right]$$
5.  **Gradient Link to Ordinary Least Squares:**
    $$\nabla_{\boldsymbol{\theta}} \mathcal{L} = \frac{1}{n} \mathbf{X}^T (\hat{\mathbf{p}} - \mathbf{y})$$
    Notice the structural parallel to the OLS normal equation gradient
    $\frac{1}{n}\mathbf{X}^T(\hat{\mathbf{y}} - \mathbf{y})$. Because
    $\hat{\mathbf{p}}$ is a non-linear function of
    $\boldsymbol{\theta}$, there is no closed-form solution
    $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^Ty$; the optimum must be
    found iteratively using Newton-Raphson or L-BFGS.

- **Model Fitting:** Fit an unregularized multivariable logistic
  regression model using all 13 standardized predictors
  (`penalty=None`).

------------------------------------------------------------------------

## Task 4: Odds Ratios & Parameter Interpretation

In linear regression, $\theta_j$ is the additive change in $y$ per unit
change in $x_j$. In logistic regression:

$$\ln\left(\frac{p}{1-p}\right) = \theta_0 + \theta_1 x_1 + \dots + \theta_j x_j + \dots + \theta_p x_p$$

Exponentiating both sides yields the multiplicative **Odds Ratio (OR)**:
$$\frac{p}{1-p} = e^{\theta_0} \cdot e^{\theta_1 x_1} \dots e^{\theta_j x_j}$$
$$\text{OR}_j = \frac{\text{Odds}(x_j + 1)}{\text{Odds}(x_j)} = e^{\theta_j}$$

- **Interpretation:** Because features were standardized ($Z$-scores), a
  1 standard deviation increase in feature $x_j$ multiplies the odds of
  heart disease by $\exp(\theta_j)$, holding all other clinical
  measurements fixed.
  - If $\theta_j > 0 \implies \text{OR}_j > 1$: the biomarker increases
    cardiovascular risk (e.g. `ca`, `oldpeak`, `exang`).
  - If $\theta_j < 0 \implies \text{OR}_j < 1$: the feature decreases
    risk / indicates protective capacity (e.g. `thalach` maximum heart
    rate achieved).
  - If $\theta_j = 0 \implies \text{OR}_j = 1$: the feature has no
    predictive association.

------------------------------------------------------------------------

## Task 5: Decision Boundaries, Thresholds & Clinical Diagnostics

A probabilistic classifier outputs $\hat{p}_i \in [0, 1]$. To make
clinical screening decisions, continuous probabilities are mapped to
discrete classes via a decision threshold $t \in [0, 1]$:
$$\hat{y}_i = \begin{cases} 1 & \text{if } \hat{p}_i \ge t \\ 0 & \text{if } \hat{p}_i < t \end{cases}$$

### Clinical Evaluation Metrics

From the $2 \times 2$ Confusion Matrix: \* **Sensitivity (Recall / True
Positive Rate):** $\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}$
— The fraction of patients with heart disease correctly flagged. \*
**Specificity (True Negative Rate):**
$\text{TNR} = \frac{\text{TN}}{\text{TN} + \text{FP}}$ — The fraction of
healthy patients correctly identified. \* **Precision (Positive
Predictive Value):**
$\text{PPV} = \frac{\text{TP}}{\text{TP} + \text{FP}}$ — The probability
that a flagged patient truly has disease.

> **Asymmetric Clinical Costs in Cardiology**
>
> In emergency cardiology and diagnostic screening: \* **False Negative
> (FN):** A patient with severe coronary artery disease is sent home
> undiagnosed, risking myocardial infarction or death. \* **False
> Positive (FP):** A healthy patient is scheduled for invasive coronary
> angiography or follow-up imaging. Because missing a cardiac event
> carries catastrophic clinical risk, screening thresholds are
> frequently adjusted below $t = 0.5$ to boost **Sensitivity**.

- **Receiver Operating Characteristic (ROC) Curve:** Plot Sensitivity
  (TPR) vs. $1 - \text{Specificity}$ (FPR) across all thresholds
  $t \in [0, 1]$.
- **Area Under the Curve (AUC):** Measures the probability that the
  model ranks a randomly chosen diseased patient higher than a randomly
  chosen healthy patient.

------------------------------------------------------------------------

# Part II: Regularization Mechanics, Ridge vs. Lasso & Tuning (Session 2 — 90 min)

## Task 6: Multicollinearity Pathology & Overfitting Audit

In clinical datasets, features often exhibit mutual collinearity
(e.g. `age`, `trestbps`, and `ca` all increase with vascular aging;
`oldpeak` and `slope` are both electrocardiographic metrics of
myocardial ischemia):

- **Matrix Conditioning:** In logistic regression, parameter covariance
  is governed by the inverse of the weighted Gram matrix (Hessian):
  $$\mathbf{H} = \nabla^2 \mathcal{L}(\boldsymbol{\theta}) = \mathbf{X}^T \mathbf{W} \mathbf{X}, \quad \text{where } \mathbf{W} = \operatorname{diag}(\hat{p}_i(1 - \hat{p}_i))$$
  When features are collinear, $\mathbf{X}^T \mathbf{W} \mathbf{X}$ is
  near-singular, driving parameter variances
  $\operatorname{Var}(\hat{\theta}_j) \to \infty$.
- **The Regularized Optimization Framework:** To tame parameter
  explosion, we augment the objective with a penalty
  $\mathcal{R}(\boldsymbol{\theta})$:
  $$\min_{\boldsymbol{\theta}} \mathcal{L}(\boldsymbol{\theta}) + \lambda \mathcal{R}(\boldsymbol{\theta}) \quad \Longleftrightarrow \quad \min_{\boldsymbol{\theta}} C \mathcal{L}(\boldsymbol{\theta}) + \mathcal{R}(\boldsymbol{\theta})$$
  where $C = \frac{1}{\lambda}$ is Scikit-Learn’s inverse regularization
  parameter:
  - **Large $C$ ($\lambda \to 0$):** Weak penalty $\to$ approaches
    unconstrained maximum likelihood.
  - **Small $C$ ($\lambda \to \infty$):** Heavy penalty $\to$ forces
    coefficients toward zero.
  - *Convention:* The intercept $\theta_0$ is unpenalized so the model
    can freely adjust baseline disease prevalence.

------------------------------------------------------------------------

## Task 7: Ridge ($L_2$) Regularization: Smooth Shrinkage

Ridge regression adds the squared Euclidean norm of the weight vector to
the loss:

$$\min_{\boldsymbol{\theta}} \mathcal{L}(\boldsymbol{\theta}) + \frac{\lambda}{2} \|\boldsymbol{\theta}_{1:p}\|_2^2 = \mathcal{L}(\boldsymbol{\theta}) + \frac{\lambda}{2} \sum_{j=1}^p \theta_j^2$$

### Key Algebraic & Geometric Properties:

1.  **Hessian Stabilization:** The Hessian becomes
    $\nabla^2 \mathcal{L}_{\text{ridge}} = \mathbf{X}^T \mathbf{W} \mathbf{X} + \lambda \mathbf{I}$.
    Adding positive $\lambda$ to the diagonal shifts all eigenvalues
    upward, guaranteeing strict positive definiteness and non-singular
    inversion.
2.  **Smooth Shrinkage:** Large parameter magnitudes are heavily
    penalized, shrinking weights continuously toward zero without
    setting any weight exactly to zero.
3.  **Collinearity Handling:** When features carry correlated clinical
    signals, Ridge encourages them to share weight smoothly rather than
    oscillating erratically.

- Fit a Ridge logistic regression model with $C = 1.0$ and compare
  coefficient magnitudes against the unregularized model.

------------------------------------------------------------------------

## Task 8: Lasso ($L_1$) Regularization: Sparsity & Biomarker Selection

Lasso replaces the $L_2$ sum of squares with the $L_1$ sum of absolute
values:

$$\min_{\boldsymbol{\theta}} \mathcal{L}(\boldsymbol{\theta}) + \lambda \|\boldsymbol{\theta}_{1:p}\|_1 = \mathcal{L}(\boldsymbol{\theta}) + \lambda \sum_{j=1}^p |\theta_j|$$

### Geometric Mechanism of Sparsity

- The constraint region for $L_1$ is a diamond/cross-polytope with sharp
  vertices on the coordinate axes.

- The elliptical loss contours of $\mathcal{L}(\boldsymbol{\theta})$
  expand outward and typically hit a vertex of the $L_1$ diamond first.

- At a vertex, one or more coordinate axes have $\theta_j \equiv 0$.

- Consequently, **Lasso performs automated clinical feature selection**,
  pruning redundant variables and extracting a sparse risk signature.

- Fit a Lasso model with $C = 0.3$ using the `saga` solver and count the
  number of non-zero features.

------------------------------------------------------------------------

## Task 9: Visualizing Regularization Paths Across 6 Orders of Magnitude

To understand how penalty strength controls model flexibility, we
compute the **regularization path**: tracking the trajectory of all 13
weights $\theta_j(C)$ as $C$ ranges from $10^{-4}$ to $10^{3}$:

- **Path Computation:** Generate 50 logarithmically spaced values:
  `C_vals = np.logspace(-4, 3, 50)`. Fit both Ridge and Lasso at each
  $C$ and record all coefficients.
- **Ridge Paths:** Observe smooth continuous decay toward zero; all 13
  lines shrink, but none equal zero.
- **Lasso Paths:** Observe piecewise-linear paths where features hit
  zero sequentially at distinct critical thresholds.
- **Sparsity Profile:** Plot the number of active features ($k$) as a
  function of $\log_{10}(C)$.

------------------------------------------------------------------------

## Task 10: 5-Fold Stratified Cross-Validation & Hyperparameter Tuning

We cannot pick the optimal penalty $C$ using training accuracy, because
training performance improves monotonically as $C \to \infty$
(overfitting). Instead, we employ **$K$-Fold Stratified
Cross-Validation** to optimize out-of-sample diagnostic ability
(ROC-AUC):

- Partition the training set into $K = 5$ stratified folds.
- Evaluate performance across the grid of candidate values $C$.
- Select $C^*$ that maximizes cross-validated ROC-AUC.
- Identify which core biomarkers survive in the optimal Lasso model.

------------------------------------------------------------------------

## Task 11: Final Out-of-Sample Evaluation & Model Critique

We now perform a comprehensive head-to-head comparison on the **held-out
20% test partition** ($n_{\text{test}} = 60$ patients never seen during
training or cross-validation):

1.  **Linear Probability Model (LPM):** Thresholded at $\hat{y} = 0.5$.
2.  **Unregularized Logistic Regression:** Maximum likelihood with all
    13 features.
3.  **Tuned Ridge Logistic Regression:** $L_2$ regularization with
    optimal $C^*_{\text{ridge}}$.
4.  **Tuned Lasso Logistic Regression:** $L_1$ regularization with
    optimal $C^*_{\text{lasso}}$.

### Evaluation Criteria:

- **Test Accuracy:** Proportion of correct classifications at threshold
  $0.5$.
- **Test ROC-AUC:** Diagnostic ranking capability independent of
  threshold.
- **Test Log-Loss (BCE):** Penalizes confident wrong probability
  predictions.
- **Active Features ($k$):** Clinical simplicity and cost of diagnostic
  testing.

> **Clinical Verdict: Parsimony in Emergency Screening**
>
> While unconstrained models require inputting all 13 clinical tests,
> the **Tuned Lasso model** achieves matching or superior diagnostic
> accuracy using only the most critical cardiovascular indicators (such
> as `ca` fluoroscopy vessels, `oldpeak` ST depression, and `thal`
> scan). Fewer required tests mean faster diagnosis in urgent cardiac
> triage, lower patient costs, and greater interpretability for
> attending physicians.

------------------------------------------------------------------------

> **Conceptual Review Questions**
>
> 1.  **Linear Regression Failure Modes:** Explain three distinct
>     mathematical reasons why Ordinary Least Squares with squared error
>     loss $(y - \hat{y})^2$ fails when applied to a binary health
>     classification task ($y \in \{0, 1\}$).
> 2.  **Geometry of Sparsity:** Geometrically explain why $L_1$ (Lasso)
>     regularization forces parameter weights strictly to zero
>     ($\theta_j \equiv 0$), whereas $L_2$ (Ridge) regularization
>     shrinks weights smoothly without setting them exactly to zero.
> 3.  **The Role of Feature Standardization:** Why is predictor
>     standardization ($Z$-scoring) mandatory before fitting Ridge or
>     Lasso models, whereas in unregularized Ordinary Least Squares it
>     simply rescales the parameter vector without affecting model fit
>     or predictions?